In [1]:
import pandas as pd
import numpy as np

# load dataset 

normal_dataset_path = '../dataset/clean/normal/normal.csv'

attack_0rtt_dataset_path = '../dataset/clean/tls/attack_0rtt_dataset.csv'
attack_heartbleed_dataset_path = '../dataset/clean/tls/attack_heartbleed_dataset.csv'


attack_cert_probe_dataset_path = '../dataset/clean/probe/attack_cert_probe_dataset.csv'
attack_crypto_probe_dataset_path = '../dataset/clean/probe/attack_crypto_probe_dataset.csv'
attack_cve_probe_dataset_path = '../dataset/clean/probe/attack_cve_probe_dataset.csv'
attack_protocol_probe_dataset_path = '../dataset/clean/probe/attack_protocol_probe_dataset.csv'

attack_goldeneye_dataset_path = '../dataset/clean/dos/attack_goldeneye_dataset.csv'
attack_hulk_dataset_path = '../dataset/clean/dos/attack_hulk_dataset.csv'
attack_rst_flood_dataset_path = '../dataset/clean/dos/attack_rst_flood_dataset.csv'
attack_slowloris_dataset_path = '../dataset/clean/dos/attack_slowloris_dataset.csv'
attack_sync_flood_dataset_path = '../dataset/clean/dos/attack_sync_flood_dataset.csv'
attack_tcp_ack_dataset_path = '../dataset/clean/dos/attack_tcp_ack_dataset.csv'
attack_tors_dataset_path = '../dataset/clean/dos/attack_tors_dataset.csv'
attack_udp_dataset_path = '../dataset/clean/dos/attack_udp_dataset.csv'




normal_dataset = pd.read_csv(normal_dataset_path)

attack_0rtt_dataset = pd.read_csv(attack_0rtt_dataset_path)
attack_cert_probe_dataset = pd.read_csv(attack_cert_probe_dataset_path)
attack_crypto_probe_dataset = pd.read_csv(attack_crypto_probe_dataset_path)
attack_cve_probe_dataset = pd.read_csv(attack_cve_probe_dataset_path)
attack_goldeneye_dataset = pd.read_csv(attack_goldeneye_dataset_path)
attack_heartbleed_dataset = pd.read_csv(attack_heartbleed_dataset_path)
attack_hulk_dataset = pd.read_csv(attack_hulk_dataset_path)
attack_protocol_probe_dataset = pd.read_csv(attack_protocol_probe_dataset_path)
attack_rst_flood_dataset = pd.read_csv(attack_rst_flood_dataset_path)
attack_slowloris_dataset = pd.read_csv(attack_slowloris_dataset_path)
attack_sync_flood_dataset = pd.read_csv(attack_sync_flood_dataset_path)
attack_tcp_ack_dataset = pd.read_csv(attack_tcp_ack_dataset_path)
attack_tors_dataset = pd.read_csv(attack_tors_dataset_path)
attack_udp_dataset = pd.read_csv(attack_udp_dataset_path)


# Randomly sample data

# Normal
normal_dataset = normal_dataset.sample(n=49000, random_state=42) 

# TLS (use full)
tls_dataset = pd.concat(
    [
        attack_0rtt_dataset,
        attack_heartbleed_dataset
    ],
    axis=0,
    ignore_index=True
)

# Probe 
probe_dataset = pd.concat(
    [
        attack_cert_probe_dataset.sample(n=1000, random_state=42),
        attack_crypto_probe_dataset.sample(n=1000, random_state=42) ,
        attack_cve_probe_dataset.sample(n=2000, random_state=42) ,
        attack_protocol_probe_dataset.sample(n=1000, random_state=42) 
    ],
    axis=0,
    ignore_index=True
)

# Dos
dos_dataset = pd.concat(
    [
        attack_goldeneye_dataset.sample(n=1000, random_state=42),
        attack_hulk_dataset.sample(n=1000, random_state=42),
        attack_rst_flood_dataset,
        attack_slowloris_dataset,
        attack_sync_flood_dataset.sample(n=1000, random_state=42),
        attack_tcp_ack_dataset.sample(n=1000, random_state=42),
        attack_tors_dataset.sample(n=1000, random_state=42),
        attack_udp_dataset
    ],
    axis=0,
    ignore_index=True
)



In [2]:
normal_df = normal_dataset
attack_df = pd.concat([tls_dataset, probe_dataset, dos_dataset], ignore_index=True)

# Combine all
dataset_df = pd.concat([normal_df, attack_df], ignore_index=True)

#  Shuffle the data
dataset_df = dataset_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Result
print("Combined shape:", dataset_df.shape)
print(dataset_df['label'].value_counts())

Combined shape: (59380, 77)
label
normal    49000
ddos       5264
probe      5000
tls         116
Name: count, dtype: int64


# IDS RNN

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score, f1_score
)

# ===== 1) Data =====
X = dataset_df.drop(columns=['label']).values           # shape: (N, 77)
y_str = dataset_df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_str)
classes = le.classes_
num_classes = len(classes)
print("Classes:", classes)

# RNNs like normalized inputs
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Splits: 70/10/20
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_full, y_tr_full, test_size=0.125, random_state=42, stratify=y_tr_full
)  # 0.125 of 0.8 ≈ 0.10

# Reshape for RNN: (batch, seq_len=77, input_size=1)
def to_seq(x):
    return torch.tensor(x, dtype=torch.float32).unsqueeze(-1)  # add feature-dim=1

X_tr_t  = to_seq(X_tr)
X_val_t = to_seq(X_val)
X_te_t  = to_seq(X_te)

y_tr_t  = torch.tensor(y_tr, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_te_t  = torch.tensor(y_te, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t),  batch_size=64, shuffle=False)

# ===== 2) Class weights (help the rare 'tls' class) =====
unique, counts = np.unique(y_tr, return_counts=True)
w_inv = {k: 1.0/v for k, v in dict(zip(unique, counts)).items()}
scale = np.mean(list(w_inv.values()))
cls_w = torch.tensor([w_inv[k]/scale for k in sorted(w_inv.keys())], dtype=torch.float32)

# ===== 3) Model: Vanilla RNN over the 77-long sequence =====
class RNNClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2,
                 nonlinearity='tanh', bidirectional=False, dropout=0.3, num_classes=4):
        super().__init__()
        self.bidirectional = bidirectional
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            nonlinearity=nonlinearity,  # 'tanh' or 'relu'
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
            bidirectional=bidirectional
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(self, x):  # x: (B, 77, 1)
        # rnn_out: (B, 77, H*[2 if bi]) ; h_n: (L*[2 if bi], B, H)
        rnn_out, h_n = self.rnn(x)
        # Take the last layer's hidden state (concatenate directions if bidirectional)
        if self.bidirectional:
            # h_n shape: (num_layers*2, B, H) -> take last layer's two directions and concat
            h_last_f = h_n[-2]  # forward
            h_last_b = h_n[-1]  # backward
            h = torch.cat([h_last_f, h_last_b], dim=1)  # (B, 2H)
        else:
            h = h_n[-1]  # (B, H)
        h = self.dropout(h)
        logits = self.fc(h)  # (B, C)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNNClassifier(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    nonlinearity='tanh',
    bidirectional=False,
    dropout=0.3,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss(weight=cls_w.to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
max_grad_norm = 2.0  # gradient clipping

# ===== 4) Train / Validate =====
epochs = 60
best_val_acc = 0.0

for epoch in range(1, epochs + 1):
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        tr_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    val_acc = correct / total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_rnn.pth")

    print(f"Epoch [{epoch}/{epochs}] "
          f"TrainLoss: {tr_loss/len(train_loader):.4f}  "
          f"ValLoss: {val_loss/len(val_loader):.4f}  "
          f"ValAcc: {val_acc:.4f}")

# ===== 5) Test =====
model.load_state_dict(torch.load("best_rnn.pth", map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        y_pred.extend(logits.argmax(dim=1).cpu().numpy())
        y_true.extend(yb.numpy())

print("TEST accuracy:", accuracy_score(y_true, y_pred))
print("TEST balanced_accuracy:", balanced_accuracy_score(y_true, y_pred))
print("TEST macro F1:", f1_score(y_true, y_pred, average="macro"))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=classes))

Classes: ['ddos' 'normal' 'probe' 'tls']
Epoch [1/60] TrainLoss: 0.7103  ValLoss: 0.4247  ValAcc: 0.9835
Epoch [2/60] TrainLoss: 0.3892  ValLoss: 0.2963  ValAcc: 0.9816
Epoch [3/60] TrainLoss: 0.3812  ValLoss: 0.3101  ValAcc: 0.9904
Epoch [4/60] TrainLoss: 0.2873  ValLoss: 0.3110  ValAcc: 0.9906
Epoch [5/60] TrainLoss: 0.3016  ValLoss: 0.3301  ValAcc: 0.9948
Epoch [6/60] TrainLoss: 0.3011  ValLoss: 0.3416  ValAcc: 0.9919
Epoch [7/60] TrainLoss: 0.2809  ValLoss: 0.4267  ValAcc: 0.9763
Epoch [8/60] TrainLoss: 0.2756  ValLoss: 0.2614  ValAcc: 0.9948
Epoch [9/60] TrainLoss: 0.2550  ValLoss: 0.2542  ValAcc: 0.9936
Epoch [10/60] TrainLoss: 0.2322  ValLoss: 0.2819  ValAcc: 0.9936
Epoch [11/60] TrainLoss: 0.2382  ValLoss: 0.3207  ValAcc: 0.9946
Epoch [12/60] TrainLoss: 0.3802  ValLoss: 0.4581  ValAcc: 0.9816
Epoch [13/60] TrainLoss: 0.2903  ValLoss: 0.3414  ValAcc: 0.9555
Epoch [14/60] TrainLoss: 0.2748  ValLoss: 0.2091  ValAcc: 0.9843
Epoch [15/60] TrainLoss: 0.2683  ValLoss: 0.2775  ValAcc: 

/tmp/ipykernel_2894697/3026931235.py:145: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_rnn.pth", map_location=device))
